# Lesson 7: Agentic Workflows

## Lab 07: Build a Multi-Step Support Workflow

So far you've built **one agent at a time**: a tool user, a knowledge-aware agent, an assistant with memory. Real AI products are rarely a single agent. They are **systems**: several steps, decisions, branches, checks and people, connected together.

This week you move **from one agent to agentic systems**.

In this activity you build a **support-desk workflow** for an online learning platform. A customer message goes in; a finished, checked reply comes out. Along the way, your workflow will:

- use an agent to **read and classify** the message,
- **route** it to the right branch,
- **look up information in parallel**,
- **draft** a reply,
- **review and revise** the draft in a loop,
- ask a **human to approve** anything involving money,
- keep everything in one shared **state**, with a full log of every step.

## Learning objectives

By the end of this activity, you should be able to:

1. Explain the difference between an autonomous agent and an agentic workflow, and when to use each.
2. Build sequential, conditional and routed workflows.
3. Use a loop with a clear exit condition.
4. Run independent steps in parallel.
5. Pass state between steps and trace what happened.
6. Add a human-in-the-loop approval step.

**Time:** about 2 hours
**You'll need:** a Gemini API key, and Weeks 1–2 completed.

> **How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code below it. Cells marked **Try it** invite you to change something and run again.

## 1. From one agent to agentic systems

In Week 1 you met the difference between a **workflow** and an **agent**:

| | Autonomous agent | Workflow |
|---|---|---|
| **Who decides the steps?** | The LLM, step by step | You, the developer, in code |
| **Flexibility** | High: can handle surprises | Lower: follows the paths you designed |
| **Predictability** | Lower: may take different paths each time | High: the same input follows the same path |
| **Cost and speed** | Harder to control | Easy to control |
| **Debugging** | Harder | Easier: you know every step |

An **agentic workflow** combines the two. **You** design the structure (the steps, the branches, the checks), and **LLMs** do the parts that need judgement inside it: reading a message, writing a reply, reviewing quality.

A good rule of thumb:

> **Put rules in code. Put judgement in the LLM.**

For example, "refunds are only allowed within 14 days" is a **rule**, so it goes in Python, where it's applied the same way every time. "Is this reply friendly and clear?" needs **judgement**, so an LLM checks it.

Most production AI systems today are agentic workflows, not fully autonomous agents, because businesses need them to be reliable.

## 2. The building blocks

Every agentic workflow is built from a small set of patterns. You'll use **all seven** in this activity.

| Pattern | What it means | Shape |
|---|---|---|
| **Sequential** | Steps run one after another; each uses the previous one's output | A → B → C |
| **Conditional** | An `if` decides whether a step runs, or ends the workflow early | A → (if X) → B |
| **Routing** | A decision sends the input down one of several branches | A → B *or* C |
| **Loop** | Repeat a step until a condition is met, with a **maximum** number of tries | A → B → back to A |
| **Parallel** | Independent steps run **at the same time** to save time | A and B together → C |
| **State** | One shared object carries all the data through every step | the "backpack" every step carries |
| **Human-in-the-loop** | A person approves, edits or rejects before an important action | … → 🧑 → … |

Here's the architecture you'll build, with each pattern labelled:

```text
               Customer message                      (input)
                      ↓
               Triage agent                          (sequential, LLM)
                      ↓
                   Route                             (routing / conditional)
              ↙               ↘
     A: Billing               B: Technical
     parallel lookups         help-doc search
     + refund policy          (no match → escalate)  (conditional early exit)
              ↘               ↙
                 Draft reply                         (LLM)
                      ↓
           Review agent ⟲ revise                     (loop, max 2 revisions)
                      ↓
           Human approval (refunds only)             (human-in-the-loop)
                      ↓
                    Result

         state = the dictionary carried through every step
```

## 3. Setup

We need `google-genai`, plus `pydantic` for structured outputs (from Week 1). Parallel execution uses Python's built-in `concurrent.futures`, so there's nothing extra to install.

In [ ]:
!pip -q install google-genai pydantic

### Add your Gemini API key

Enter your key when prompted. It won't be shown on screen or saved in the notebook. **Never paste your API key directly into a code cell.**

In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("API key loaded.")

In [ ]:
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import date, timedelta
from typing import Literal, Optional

from google import genai
from google.genai import types
from pydantic import BaseModel, Field

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemini-3.8-flash"

print("Gemini client is ready. Model:", MODEL)

Every LLM step in this workflow makes **one** call to Gemini, so we use one small helper. With a `schema`, it returns structured output (a Pydantic object); without one, it returns plain text. Like Lessons 5 and 6, it retries automatically when Gemini is busy.

In [ ]:
def ask_llm(prompt: str, system: str, schema=None):
    # Call Gemini once. With a schema, returns a parsed object; otherwise returns text. Retries when busy.
    config = types.GenerateContentConfig(system_instruction=system)
    if schema is not None:
        config.response_mime_type = "application/json"
        config.response_schema = schema
    for attempt in range(4):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
            return response.parsed if schema is not None else response.text.strip()
        except Exception as e:
            busy = any(code in str(e) for code in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED"))
            if busy and attempt < 3:
                time.sleep(2 ** attempt)
            else:
                raise


print("ask_llm() is ready.")

## 4. The project: a support desk

Our platform sells online courses. Customers write in about two kinds of things:

- **Billing:** refunds, charges, invoices. This needs account data and a refund policy.
- **Technical:** videos not playing, passwords, certificates. This needs the help-centre articles.

Below is a tiny pretend "database": three customers, two orders and five help articles. Order **A-1001** was bought 5 days ago (refundable); **A-1002** was bought 40 days ago (not refundable).

In [ ]:
REFUND_WINDOW_DAYS = 14       # refund policy: within 14 days of purchase...
MAX_PROGRESS_FOR_REFUND = 30  # ...and less than 30% of the course completed

TODAY = date.today()

CUSTOMERS = {
    "C001": {"name": "Lena Fischer", "plan": "Pro"},
    "C002": {"name": "Kenji Watanabe", "plan": "Basic"},
    "C003": {"name": "Maria Silva", "plan": "Pro"},
}

ORDERS = {
    "A-1001": {"customer_id": "C001", "course": "Python for Data Analysis", "price_usd": 49,
               "purchased": str(TODAY - timedelta(days=5)), "progress_pct": 10},
    "A-1002": {"customer_id": "C002", "course": "Intro to Machine Learning", "price_usd": 79,
               "purchased": str(TODAY - timedelta(days=40)), "progress_pct": 85},
}

HELP_DOCS = [
    {"title": "Videos won't play or keep buffering",
     "text": "Refresh the page, switch the video quality to 480p, and close other tabs that use the internet. "
             "If it still freezes, clear your browser cache or try a different browser.",
     "keywords": ["video", "buffering", "freezing", "loading", "playing", "stream", "slow", "lag"]},
    {"title": "Resetting your password",
     "text": "Click 'Forgot password' on the login page and follow the link we email you. The link expires after 1 hour.",
     "keywords": ["password", "login", "reset", "forgot", "locked", "sign"]},
    {"title": "Downloading your certificate",
     "text": "Certificates unlock when you finish 100% of a course. Open the course, choose 'Certificate', then 'Download PDF'.",
     "keywords": ["certificate", "download", "pdf", "complete", "finish", "proof"]},
    {"title": "Changing your account email",
     "text": "Go to Settings > Account > Email, enter the new address, and confirm it from the email we send you.",
     "keywords": ["email", "account", "change", "settings", "profile", "address"]},
    {"title": "Learning offline with the mobile app",
     "text": "In the mobile app, tap the download icon next to a lesson to watch it offline. Downloads last 30 days.",
     "keywords": ["mobile", "app", "offline", "phone", "download", "tablet", "android", "iphone"]},
]

print("Customers:", list(CUSTOMERS))
print("Orders:   ", list(ORDERS))
print("Help docs:", [d["title"] for d in HELP_DOCS])

## 5. State

In a workflow, every step needs to know what earlier steps found. Instead of passing ten separate variables around, we keep **one dictionary, the state**, and every step reads from it and writes to it.

Think of it as a backpack the message carries through the workflow. Each step opens it, uses what it needs, and adds what it found.

| Key | Filled in by |
|---|---|
| `message`, `customer_id` | The input |
| `triage` | Step 1: the triage agent |
| `route` | Step 2: the routing decision |
| `context` | The branch: customer, order, policy or help docs |
| `draft`, `reviews`, `revisions` | The draft and the review loop |
| `needs_approval`, `approval` | Human-in-the-loop |
| `status`, `final_reply` | The result |
| `trace` | Every step adds a line: what ran, how long it took, and what happened |

The `trace` is how you debug a workflow. When something goes wrong, it tells you exactly which step did what.

In [ ]:
def new_state(message: str, customer_id: str) -> dict:
    return {
        "message": message,          # input
        "customer_id": customer_id,
        "triage": None,              # set by triage()
        "route": None,               # set by route()
        "context": {},               # facts gathered by the branch
        "draft": None,               # the reply being worked on
        "reviews": [],               # every review from the loop
        "revisions": 0,
        "needs_approval": False,     # True for refunds
        "approval": None,            # the human's decision
        "status": "in_progress",     # becomes sent / escalated / held
        "final_reply": None,
        "trace": [],                 # a log of every step, for debugging
    }


def log_step(state: dict, step: str, started: float, note: str = "") -> None:
    # Record one step in the trace, with how long it took.
    seconds = round(time.time() - started, 2)
    state["trace"].append({"step": step, "seconds": seconds, "note": note})
    print(f"  ✓ {step:<16} {seconds:>5.2f}s  {note}")


example = new_state("My videos won't load.", "C003")
print(json.dumps(example, indent=2))

## 6. Sequential workflows: the triage agent

A **sequential** workflow is a chain: each step takes the previous step's output as its input.

The first step is an LLM **triage agent**. It reads the message and returns **structured output** (from Week 1), so the next step can rely on the exact fields:

- `category`: `billing` or `technical`
- `urgency`: `low`, `medium` or `high`
- `order_id`: if the customer mentioned one
- `summary`: one sentence

Every step function follows the same pattern: **take the state, do one job, update the state, log it, return the state.**

In [ ]:
class Triage(BaseModel):
    category: Literal["billing", "technical"] = Field(
        description="billing = payments, refunds, charges, invoices. technical = anything about using the platform.")
    urgency: Literal["low", "medium", "high"]
    order_id: Optional[str] = Field(None, description="Order ID like A-1001 if the customer mentions one, else null")
    summary: str = Field(description="One sentence describing what the customer needs")


TRIAGE_SYSTEM = "You are the triage agent for an online learning platform's support desk. Read the customer message and classify it."


def triage(state: dict) -> dict:
    started = time.time()
    result = ask_llm(state["message"], TRIAGE_SYSTEM, schema=Triage)
    state["triage"] = result.model_dump()
    log_step(state, "triage", started, f"{result.category}, urgency {result.urgency}")
    return state


for message in ["I was charged twice for order A-1001!",
                "How do I download my certificate?"]:
    s = triage(new_state(message, "C001"))
    print("   ", s["triage"], "\n")

## 7. Conditional workflows and routing

**Routing** sends the input down one of several branches. Notice who does what:

- The **LLM** made a judgement: "this message is about billing".
- **Code** makes the decision: "billing messages go to the billing branch".

Keeping the decision in code means it's predictable, easy to test, and easy to change.

A **conditional** step is any `if` that changes what happens next. You'll see one later: if no help article matches, the workflow **ends early** and escalates to a person.

In [ ]:
def route(state: dict) -> str:
    # Decide which branch handles this message. Returns "billing" or "technical".
    started = time.time()
    category = state["triage"]["category"]
    branch = "billing" if category == "billing" else "technical"
    state["route"] = branch
    log_step(state, "route", started, f"→ {branch} branch")
    return branch


for message in ["Can I get a refund for order A-1002?",
                "The video player just shows a black screen."]:
    s = triage(new_state(message, "C001"))
    print("    Branch:", route(s), "\n")

## 8. Parallel execution

The billing branch needs two pieces of information: the **customer** and the **order**. In a real system, each lookup is a call to a database or API that takes time. Ours pretend to take 1 second each.

The two lookups don't depend on each other, so there's no reason to wait for one before starting the other. We can run them **in parallel**.

Python's `ThreadPoolExecutor` does this:

- `executor.submit(function, argument)` **starts** a job straight away and returns a "future", a promise of a result.
- `future.result()` **waits** for that job to finish and gives you its result.

Run the cell and compare the times.

In [ ]:
LOOKUP_DELAY = 1.0   # simulated database delay, in seconds


def get_customer(customer_id: str) -> dict:
    time.sleep(LOOKUP_DELAY)   # pretend this is a slow database call
    return CUSTOMERS.get(customer_id, {"error": f"No customer {customer_id}"})


def get_order(order_id: Optional[str]) -> dict:
    time.sleep(LOOKUP_DELAY)   # pretend this is a slow database call
    if not order_id:
        return {"error": "No order ID given"}
    return ORDERS.get(order_id.upper(), {"error": f"No order {order_id}"})


def gather_billing_context(customer_id: str, order_id: Optional[str]) -> tuple[dict, dict]:
    # Look up the customer and the order AT THE SAME TIME. Returns (customer, order).
    with ThreadPoolExecutor() as executor:
        customer_job = executor.submit(get_customer, customer_id)
        order_job = executor.submit(get_order, order_id)
        return customer_job.result(), order_job.result()


started = time.time()
get_customer("C001")
get_order("A-1001")
print(f"One after the other: {time.time() - started:.2f}s")

started = time.time()
customer, order = gather_billing_context("C001", "A-1001")
print(f"In parallel:         {time.time() - started:.2f}s\n")

print("Customer:", customer)
print("Order:   ", order)

Same results, about half the time. With five lookups, parallel would still take about 1 second instead of 5.

**Only run steps in parallel when they don't depend on each other.** Drafting a reply needs the lookup results, so drafting must wait.

> **Try it:** add a third lookup (for example `get_customer("C002")`) to both versions. How do the times change?

## 9. Branch A: billing

The billing branch does three things:

1. Look up the customer and order **in parallel**.
2. Check the **refund policy in code**, following "rules in code, judgement in the LLM". The policy is: within 14 days of purchase, and under 30% of the course completed.
3. Ask the LLM to **draft** a reply using only those facts.

If a refund is allowed, `needs_approval` is set to `True`. No money moves without a person agreeing, which is the human-in-the-loop step later.

In [ ]:
def check_refund_policy(order: dict) -> dict:
    # The refund rule lives in code, not in the LLM, so it's applied the same way every time.
    if "error" in order:
        return {"eligible": False, "reason": "We couldn't find the order, so we need the order ID."}
    days = (TODAY - date.fromisoformat(order["purchased"])).days
    if days > REFUND_WINDOW_DAYS:
        return {"eligible": False, "reason": f"Purchased {days} days ago; refunds are only possible within {REFUND_WINDOW_DAYS} days."}
    if order["progress_pct"] >= MAX_PROGRESS_FOR_REFUND:
        return {"eligible": False, "reason": f"{order['progress_pct']}% of the course is completed; refunds need under {MAX_PROGRESS_FOR_REFUND}%."}
    return {"eligible": True, "reason": f"Purchased {days} days ago with {order['progress_pct']}% progress: within policy.",
            "amount_usd": order["price_usd"]}


print("A-1001:", check_refund_policy(ORDERS["A-1001"]))
print("A-1002:", check_refund_policy(ORDERS["A-1002"]))

In [ ]:
WRITER_SYSTEM = """You write replies for an online learning platform's support desk.
Rules: greet the customer by first name, stay under 120 words, be warm and professional,
use ONLY the facts provided, never promise anything the facts don't support, and end with a clear next step.
Write only the reply text."""


def billing_branch(state: dict) -> dict:
    started = time.time()
    customer, order = gather_billing_context(state["customer_id"], state["triage"]["order_id"])
    policy = check_refund_policy(order)
    state["context"] = {"customer": customer, "order": order, "refund_policy": policy}
    state["needs_approval"] = policy["eligible"]   # a person approves every refund
    log_step(state, "billing lookups", started, f"refund eligible: {policy['eligible']}")

    started = time.time()
    prompt = (f"Customer message: {state['message']}\n\nFacts:\n{json.dumps(state['context'], indent=2)}\n\n"
              "If the refund is eligible, say the refund has been requested and will be confirmed shortly. "
              "If not, explain the reason kindly.")
    state["draft"] = ask_llm(prompt, WRITER_SYSTEM)
    log_step(state, "draft reply", started)
    return state


demo_billing = triage(new_state("Hi, can I get a refund for order A-1001? It's not what I expected.", "C001"))
route(demo_billing)
demo_billing = billing_branch(demo_billing)
print("\nDraft:\n", demo_billing["draft"])

## 10. Branch B: technical

The technical branch searches the help articles. It uses simple keyword matching, as in Lesson 6: an article needs **at least 2 matching keywords** to count.

This branch also has a **conditional early exit**. If no article matches, the workflow doesn't guess. It sends a short holding reply, sets the status to `escalated`, and skips the rest of the workflow so a person can take over.

In [ ]:
STOPWORDS = {"the", "and", "for", "with", "you", "your", "can", "what", "how", "my", "keep", "keeps",
             "on", "it", "is", "are", "do", "does", "i", "to", "a", "an", "of", "in", "course", "help", "please"}


def search_help_docs(text: str, min_matches: int = 2) -> list[dict]:
    # Return help docs that share at least `min_matches` keywords with the text, best first.
    words = {w.rstrip("s") for w in re.findall(r"[a-z]+", text.lower()) if w not in STOPWORDS}
    scored = []
    for doc in HELP_DOCS:
        score = len(words & {k.rstrip("s") for k in doc["keywords"]})
        if score >= min_matches:
            scored.append((score, doc))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [doc for score, doc in scored]


def technical_branch(state: dict) -> dict:
    started = time.time()
    docs = search_help_docs(state["message"] + " " + state["triage"]["summary"])
    customer = CUSTOMERS.get(state["customer_id"], {})
    state["context"] = {"customer": customer, "help_docs": docs}
    log_step(state, "search help docs", started, f"{len(docs)} article(s) found")

    if not docs:
        # Conditional early exit: nothing to answer with, so a person takes over.
        first_name = customer.get("name", "there").split()[0]
        state["draft"] = (f"Hi {first_name}, thanks for your message. I've passed your question to our "
                          "technical team, and they'll reply within one working day.")
        state["status"] = "escalated"
        log_step(state, "escalate", time.time(), "no matching help article")
        return state

    started = time.time()
    prompt = f"Customer message: {state['message']}\n\nFacts:\n{json.dumps(state['context'], indent=2)}"
    state["draft"] = ask_llm(prompt, WRITER_SYSTEM)
    log_step(state, "draft reply", started)
    return state


for message in ["My videos keep buffering and freezing.", "Can I pay with cryptocurrency?"]:
    print(message, "→", [d["title"] for d in search_help_docs(message)])

In [ ]:
demo_tech = triage(new_state("The course videos keep buffering on my laptop. What can I do?", "C003"))
route(demo_tech)
demo_tech = technical_branch(demo_tech)
print("\nDraft:\n", demo_tech["draft"])

## 11. Loops: review and revise

LLM drafts are good, but not always good enough. So we add a second agent, a **reviewer**, which checks each draft against clear rules and returns structured output: `approved` (true or false) and `feedback`.

If the draft isn't approved, the writer **revises** it using the feedback, and the reviewer checks again. That's a **loop**.

Every loop needs an **exit condition**, or it could run forever (remember Lesson 4's stopping rules). This loop stops when:

1. the reviewer approves, **or**
2. it has revised `MAX_REVISIONS` times. Then a person checks the draft instead (`needs_approval = True`).

```text
draft → review ── approved ──────────────────────→ done
          │
          └─ not approved → revisions left? ── no ─→ human checks it
                                  │
                                 yes → revise → review again
```

In [ ]:
MAX_REVISIONS = 2   # loop limit


class Review(BaseModel):
    approved: bool
    feedback: str = Field(description="If not approved: exactly what to fix. If approved: 'OK'.")


REVIEW_SYSTEM = """You are a strict quality reviewer for support replies. Approve the draft only if ALL are true:
1. It greets the customer by first name.
2. It is under 120 words.
3. Every claim matches the facts provided. It promises nothing the facts don't support.
4. It ends with a clear next step for the customer.
5. The tone is warm and professional."""


def review_draft(state: dict) -> Review:
    prompt = f"Facts:\n{json.dumps(state['context'], indent=2)}\n\nDraft reply:\n{state['draft']}"
    return ask_llm(prompt, REVIEW_SYSTEM, schema=Review)


def revise_draft(state: dict, feedback: str) -> str:
    prompt = (f"Customer message: {state['message']}\n\nFacts:\n{json.dumps(state['context'], indent=2)}\n\n"
              f"Current draft:\n{state['draft']}\n\nReviewer feedback:\n{feedback}\n\nRewrite the draft to fix the feedback.")
    return ask_llm(prompt, WRITER_SYSTEM)


def review_loop(state: dict) -> dict:
    # Review the draft; if it isn't approved, revise and review again, at most MAX_REVISIONS times.
    while True:
        started = time.time()
        review = review_draft(state)
        state["reviews"].append(review.model_dump())
        log_step(state, "review", started, "approved" if review.approved else f"needs changes: {review.feedback[:60]}")

        if review.approved:
            break
        if state["revisions"] >= MAX_REVISIONS:
            log_step(state, "review limit", time.time(), "max revisions reached; a person will check it")
            state["needs_approval"] = True
            break

        started = time.time()
        state["draft"] = revise_draft(state, review.feedback)
        state["revisions"] += 1
        log_step(state, "revise", started, f"revision {state['revisions']}")
    return state


print("Review loop is ready.")

Let's test the loop with a deliberately **bad** draft: no greeting, an over-excited tone, and a promise the facts don't support (the refund isn't confirmed yet).

In [ ]:
demo_loop = dict(demo_billing)          # reuse the billing example's facts
demo_loop["reviews"], demo_loop["revisions"], demo_loop["trace"] = [], 0, []
demo_loop["draft"] = "REFUND DONE!!! Money is already back in your account. Bye."

demo_loop = review_loop(demo_loop)

print("\nRevisions:", demo_loop["revisions"])
print("Final draft:\n", demo_loop["draft"])

The reviewer should reject the bad draft, the writer should fix it, and the loop should stop once it's approved, or after 2 revisions.

> **Try it:** set `MAX_REVISIONS = 0` and run the two cells above again. What happens to `needs_approval`?

## 12. Human-in-the-loop

Some actions are too important to leave to an AI alone: sending money, deleting data, sending a message to thousands of people. For these, the workflow **pauses** and asks a person.

The person has three choices:

| Decision | What happens |
|---|---|
| **Approve** | The draft is sent as it is |
| **Edit** | The person's version is sent instead |
| **Reject** | Nothing is sent; the status becomes `held` |

The approval step takes an `approver` function, so we can swap who decides:

- `console_approver` asks a real person by typing (`input()`),
- `auto_approver` always approves, which is used for automatic tests.

This is the same "least privilege and human approval" guardrail from the Lesson 5 safety document.

In [ ]:
def console_approver(state: dict) -> dict:
    # Ask a real person in the terminal. Returns {"decision": "approve"|"edit"|"reject", "text": ...}.
    print("\n" + "-" * 60)
    print("HUMAN APPROVAL NEEDED")
    policy = state["context"].get("refund_policy", {})
    if policy.get("eligible"):
        print(f"Refund: ${policy['amount_usd']}  ({policy['reason']})")
    print(f"\nDraft reply:\n{state['draft']}\n")
    choice = input("Approve (a), edit (e) or reject (r)? ").strip().lower()
    if choice.startswith("e"):
        return {"decision": "edit", "text": input("Type the new reply: ").strip()}
    if choice.startswith("r"):
        return {"decision": "reject", "text": None}
    return {"decision": "approve", "text": None}


def auto_approver(state: dict) -> dict:
    # Used by --test so the tests can run without anyone typing.
    return {"decision": "approve", "text": None}


def human_approval(state: dict, approver=console_approver) -> dict:
    started = time.time()
    decision = approver(state)
    state["approval"] = decision["decision"]

    if decision["decision"] == "approve":
        state["final_reply"] = state["draft"]
    elif decision["decision"] == "edit":
        state["final_reply"] = decision["text"]
    else:  # reject
        state["final_reply"] = None
        state["status"] = "held"

    log_step(state, "human approval", started, decision["decision"])
    return state


print("Human-in-the-loop step is ready.")

**Try it:** run the cell below. You'll see the refund and the draft. Type `a` to approve, `e` to edit, or `r` to reject.

In [ ]:
demo_human = dict(demo_billing)
demo_human["trace"] = []
demo_human = human_approval(demo_human, console_approver)

print("\nApproval:", demo_human["approval"])
print("Status:  ", demo_human["status"])
print("Will send:", demo_human["final_reply"])

## 13. Putting it all together

Now we connect every step into one workflow. Read `run_workflow()` slowly. It reads like the architecture diagram, and every pattern appears in it:

- **Sequential:** triage, then route, then the branch, then review.
- **Routing:** `if branch == "billing"`.
- **Conditional:** escalated messages skip review and approval.
- **Loop:** inside `review_loop`.
- **Parallel:** inside `billing_branch`.
- **Human-in-the-loop:** only when `needs_approval` is true.
- **State:** passed into and returned from every step.

In [ ]:
def finalize(state: dict) -> dict:
    if state["status"] == "in_progress":
        state["status"] = "sent"
    if state["final_reply"] is None and state["status"] in ("sent", "escalated"):
        state["final_reply"] = state["draft"]
    return state


def run_workflow(message: str, customer_id: str, approver=console_approver) -> dict:
    print(f"\nCustomer {customer_id}: {message}")
    state = new_state(message, customer_id)

    state = triage(state)                     # 1. sequential: the agent reads the message
    branch = route(state)                     # 2. conditional: decide where it goes

    if branch == "billing":                   # 3. branch A or branch B
        state = billing_branch(state)
    else:
        state = technical_branch(state)

    if state["status"] != "escalated":        # escalated messages skip straight to the end
        state = review_loop(state)            # 4. loop: review and revise
        if state["needs_approval"]:
            state = human_approval(state, approver)   # 5. human-in-the-loop

    state = finalize(state)
    print(f"\n  Status: {state['status'].upper()}")
    if state["final_reply"]:
        print(f"  Reply:\n{state['final_reply']}")
    return state


def show_trace(state):
    # Print the workflow's trace as a table.
    print(f"\n{'step':<18}{'seconds':>8}  note")
    for t in state["trace"]:
        print(f"{t['step']:<18}{t['seconds']:>8.2f}  {t['note']}")
    print(f"{'TOTAL':<18}{sum(t['seconds'] for t in state['trace']):>8.2f}")

**Try it:** run the full workflow on a refund request. You'll be asked to approve it.

In [ ]:
result = run_workflow(
    "Hello! I bought Python for Data Analysis a few days ago, order A-1001, but I'd like a refund please.",
    "C001",
    approver=console_approver,
)
show_trace(result)

The trace shows every step, how long it took and what happened. When a workflow misbehaves, this is the first place to look.

> **Try it:** change the message and customer (C001, C002 or C003) and run it again. Can you make it go down the technical branch? Can you make it escalate?

## 14. Test the workflow

A workflow should handle every path correctly. These four test cases cover them all:

| Test | Path it tests |
|---|---|
| Eligible refund | Billing → parallel lookups → review loop → **human approval** → sent |
| Refund outside policy | Billing → policy says no → review loop → sent (**no approval needed**) |
| Help article found | Technical → help article → review loop → sent |
| No help article | Technical → **early exit** → escalated |

We use `auto_approver` so the tests run without anyone typing.

In [ ]:
TEST_CASES = [
    {"name": "eligible refund", "customer_id": "C001",
     "message": "Hi, I bought Python for Data Analysis last week (order A-1001) but it isn't what I expected. Can I get a refund?",
     "expect": {"route": "billing", "needs_approval": True, "status": "sent"}},
    {"name": "refund outside policy", "customer_id": "C002",
     "message": "I want my money back for order A-1002. The course was too basic for me.",
     "expect": {"route": "billing", "needs_approval": False, "status": "sent"}},
    {"name": "help article found", "customer_id": "C003",
     "message": "The course videos keep buffering and freezing on my laptop. What can I do?",
     "expect": {"route": "technical", "status": "sent"}},
    {"name": "no help article: escalate", "customer_id": "C003",
     "message": "Can I connect my account to my smartwatch to track my study time?",
     "expect": {"route": "technical", "status": "escalated"}},
]


checks = []
test_runs = []

for case in TEST_CASES:
    print("\n" + "=" * 60 + f"\nTEST: {case['name']}")
    state = run_workflow(case["message"], case["customer_id"], approver=auto_approver)
    test_runs.append(state)
    for key, expected in case["expect"].items():
        checks.append((f"{case['name']}: {key} = {expected}", state.get(key) == expected, state.get(key)))
    checks.append((f"{case['name']}: revisions within limit", state["revisions"] <= MAX_REVISIONS, state["revisions"]))

print("\n" + "=" * 60 + "\nCHECKS")
for name, ok, actual in checks:
    print(("PASS " if ok else "CHECK"), name, "" if ok else f"(got {actual})")
print(f"\n{sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed.")

### What to look for

- In the **eligible refund** test, find the `human approval` line in the output. In the **outside policy** test, it shouldn't appear at all.
- Did any draft need a **revision**? Read the reviewer's feedback in the `review` lines.
- In the **escalate** test, there's no draft, review or approval step. The conditional exit skipped them.

The LLM steps can behave a little differently on each run. If a check shows `CHECK`, use the trace to find the step that went a different way.

## 15. Challenge

### Part A: Add a third route

Customers also send **feedback**: compliments, complaints about a course, and feature requests. Add a third branch:

1. Add `"feedback"` to the `category` options in the `Triage` schema, and describe it in the field description.
2. Update `route()` to return `"feedback"`.
3. Write `feedback_branch(state)`. It should save the message to a `feedback.json` file and draft a short thank-you reply with the LLM.
4. Update `run_workflow()` to call it.
5. Test it with: *"I love the new dark mode! Could you add subtitles in Spanish too?"*

### Part B: An urgency rule

Any message the triage agent marks as `urgency: "high"` should **always** go to a human for approval, whichever branch it took. Add this rule in code (where does it belong?), then test it with an angry, urgent message.

### Stretch: Parallel reviewers

Replace the single reviewer with **two reviewers that run in parallel**: one checks **tone**, the other checks **accuracy against the facts**. The draft is approved only if both approve. Combine their feedback when revising. You'll use parallel execution *inside* a loop.

In [ ]:
# Part A: write feedback_branch() here, then update Triage, route() and run_workflow() above.

def feedback_branch(state: dict) -> dict:
    started = time.time()
    # TODO: append {"customer_id", "message", "date"} to feedback.json (create the file if it doesn't exist)
    # TODO: draft a short thank-you reply with ask_llm(...) and WRITER_SYSTEM, and store it in state["draft"]
    raise NotImplementedError("Challenge Part A: complete feedback_branch()")
    log_step(state, "feedback saved", started)
    return state

## Recap

| Pattern | Where it is in your workflow |
|---|---|
| **Sequential** | `run_workflow()`: triage → route → branch → review → approval |
| **Conditional** | Escalated messages skip review; approval only when `needs_approval` |
| **Routing** | `route()` sends billing and technical messages to different branches |
| **Loop** | `review_loop()`: review → revise, up to `MAX_REVISIONS` |
| **Parallel** | `gather_billing_context()`: two lookups at once |
| **State** | The dictionary carried through every step, with its `trace` |
| **Human-in-the-loop** | `human_approval()`: approve, edit or reject before refunds go out |

You've built an **agentic system**: several LLM roles (triage, writer, reviewer) and plain code (routing, policy, lookups) working together, with a person in charge of the important decisions.

**Next:** complete the starter file `lesson_07_workflow.py`, run its tests, and submit your work. See `README.md` for the submission checklist.